# Anti-Spoofing Model Test
Compare Silent-FAS and deepface anti-spoofing performance on real vs spoof images.

In [1]:
import os
import sys
import cv2

# Add Silent-FAS to path
fas_root = os.path.abspath("../third_party/Silent-Face-Anti-Spoofing")
sys.path.insert(0, fas_root)
os.chdir(fas_root)

## 1. deepface Anti-Spoofing Test

In [2]:
from deepface import DeepFace

img_real = cv2.imread(os.path.abspath("../../notebooks/test_real.jpg"))
results = DeepFace.extract_faces(img_real, anti_spoofing=True, enforce_detection=False)
print(f"[Real Image] is_real: {results[0]['is_real']}, score: {results[0]['antispoof_score']:.4f}")


[Real Image] is_real: True, score: 0.9982


In [3]:
img_spoof = cv2.imread(os.path.abspath("../../notebooks/test_spoof.jpg"))
results = DeepFace.extract_faces(img_spoof, anti_spoofing=True, enforce_detection=False)
print(f"[Spoof Image] is_real: {results[0]['is_real']}, score: {results[0]['antispoof_score']:.4f}")

[Spoof Image] is_real: False, score: 0.9910


## 2. Silent-FAS Anti-Spoofing Test

In [4]:
from src.anti_spoof_predict import AntiSpoofPredict

predictor = AntiSpoofPredict(device_id=0)
model_dir = "resources/anti_spoof_models"
model_paths = sorted([os.path.join(model_dir, f) for f in os.listdir(model_dir) if f.endswith(".pth")])

def run_silent_fas(image):
    bbox = predictor.get_bbox(image)
    x, y, w, h = bbox
    face = cv2.resize(image[y:y+h, x:x+w], (80, 80))
    total = 0.0
    for path in model_paths:
        result = predictor.predict(face, path)
        total += result[0][1]
    return total / len(model_paths)

score = run_silent_fas(img_real)
print(f"[Real Image] score: {score:.4f}, is_real: {score >= 0.5}")

[Real Image] score: 0.8517, is_real: True


In [5]:
score = run_silent_fas(img_spoof)
print(f"[Spoof Image] score: {score:.4f}, is_real: {score >= 0.5}")

[Spoof Image] score: 0.0049, is_real: False


## 3. Results Summary

| Model | Real Image | Spoof Image | Correct? |
|-------|-----------|-------------|----------|
| deepface | REAL (99.8%) | FALSE (99.1%) | 2/2 |
| Silent-FAS | REAL (85.2%) | SPOOF (0.5%) | 2/2 |

Both models correctly classify real and spoof images.
Silent-FAS shows much stronger confidence on spoof detection (0.5% vs 99.1%).